In [15]:
!pip install faiss-cpu openai


Defaulting to user installation because normal site-packages is not writeable
  Using cached anyio-4.8.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jiter-0.9.0-cp312-cp312-win_amd64.whl.metadata (5.3 kB)
  Using cached pydantic-2.10.6-py3-none-any.whl.metadata (30 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached httpcore-1.0.7-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.14.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.27.2-cp312-cp312-win_amd64.whl.metadata (6.7 kB)
   ---------------------------------------- 0.0/13.7 MB ? eta -:--:--
   ----------- ---------------------------- 3.9/13.7 MB 19.5 MB/s eta 0:00:01
   ----------------------------- ---------- 10.0/


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: C:\Users\r2com\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import json
import numpy as np
import faiss
import os
from openai import OpenAI

# 모든 JSON 데이터 로드
json_files = [f for f in os.listdir("C:/Users/r2com/Documents/GitHub/BeBaek/ChefBot/data/recipes") if f.endswith(".json")]

all_recipes = []
for file in json_files:
    with open(f"C:/Users/r2com/Documents/GitHub/BeBaek/ChefBot/data/recipes/{file}", "r", encoding="utf-8") as f:
        data = json.load(f)
        all_recipes.extend(data)

print(f"총 {len(all_recipes)} 개의 레시피 데이터 로드 완료")

# OpenAI API 키 설정
API_KEY = '' # 깃헙 반영을 위해 공백 처리  
client = OpenAI(api_key=API_KEY)

# 벡터화할 텍스트 데이터 준비
texts = [f"{recipe['name']} {recipe['ingredients']} {' '.join(recipe['recipe'])}" for recipe in all_recipes]

# OpenAI Embedding 생성 (Batch 처리)
def get_embeddings(texts, batch_size=100):
    embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        try:
            response = client.embeddings.create(
                input=batch, 
                model="text-embedding-ada-002"
            )
            batch_embeddings = [item.embedding for item in response.data] 
            embeddings.extend(batch_embeddings)
        except Exception as e:
            print(f"OpenAI API 호출 오류 발생: {e}")

    return np.array(embeddings)

# 모든 레시피 데이터 임베딩 생성
embeddings = get_embeddings(texts)

# FAISS 벡터 DB 구축
d = embeddings.shape[1]  # 벡터 차원 수
index = faiss.IndexFlatL2(d)  # L2 거리 기반 벡터 검색 인덱스
index.add(embeddings)  # 벡터 추가

# FAISS 인덱스 저장
faiss.write_index(index, "./faiss_index/recipes_faiss.index")

print("벡터화 완료 & FAISS 저장 완료")

총 2071 개의 레시피 데이터 로드 완료
벡터화 완료 & FAISS 저장 완료


In [2]:
import faiss

# 저장된 FAISS 인덱스 로드
index_path = "./faiss_index/recipes_faiss.index"
index = faiss.read_index(index_path)

# 벡터 개수 확인
print(f"FAISS 인덱스 로드 완료, 저장된 벡터 개수: {index.ntotal}")

FAISS 인덱스 로드 완료, 저장된 벡터 개수: 2071


In [3]:
import numpy as np
import faiss

# 저장된 FAISS 인덱스 로드
index = faiss.read_index(index_path)

# 테스트용 랜덤 벡터 생성 (벡터 차원 수 맞추기)
d = index.d  # 벡터 차원 수
test_vector = np.random.random((1, d)).astype('float32')

# 가장 가까운 3개 벡터 찾기
D, I = index.search(test_vector, 3)

print(f"검색된 벡터의 인덱스: {I[0]}")
print(f"검색된 벡터의 거리: {D[0]}")


검색된 벡터의 인덱스: [1002  491  351]
검색된 벡터의 거리: [496.41165 496.51218 496.52203]


In [4]:
# 검색된 레시피 확인
search_results = [all_recipes[i] for i in I[0]]

print("검색된 레시피:")
for i, recipe in enumerate(search_results):
    print(f"{i+1}. {recipe['name']} - 재료: {recipe['ingredients']} - 레시피: {recipe['recipe']}")

검색된 레시피:
1. 삼겹살 스테이크 - 재료: 삼겹살 300g, 소금 약간, 후추 약간, 돈까스소스 3T, 굴소스 1T, 맛술 2T, 물 2T, 올리고당 1T, 다진마늘 0.5T - 레시피: ['1. 고기에 소금과 후추를 뿌려서 준비해주세요.', '2. 돈까스소스 3T, 굴소스 1T, 맛술 2T, 물 2T, 올리고당 1T, 다진마늘 0.5T를 넣고 섞어서 소스를 만들어주세요.', '3. 달군 팬에 삼겹살을 앞, 뒤로 노릇노릇 구워준 후, 기름을 살짝 제거해주세요.', '4. 소스를 넣고 걸쭉하게 끓여주시면 완성이에요. 간단하면서도 맛도 좋아요!', '5. 집에 있는 어린잎도 함께~ 스테이크라고 했지만 밥과 먹으면 더 맛있어요 :D 역시 삼겹살엔 밥이 최고인 것 같아요.']
2. 동남아음식 - 재료: 공심채 1단, 다진마늘 1ts, 홍고추 1개, 굴소스 2T, 멸치액젓 1T - 레시피: ['1. 조금은 생소할 수 있는 야채 이게 바로 공심채 입니다. 농협 하나로 마트에서 구매를 해놓고  냉장고에 조금 오래 보관을 했더니 시들시들하네요.. 먹기 좋게 한입 사이즈로 잘라줍니다.', '2. 다진마늘 1ts, 홍고추 1개를 잘개 썰어서 기름에 달달 볶아줍니다.', '3. 그리고 공심채도 같이 볶아줍니다. 식감을 위해서 잎사귀 부분은 나중에 볶지만, 저는 잎사귀의 눅눅해진 맛을 좋아하기 때문에 한꺼번에 다 볶았습니다.', '4. 공심채의 숨이 죽으면 멸치액젓 1TS, 굴소스 2TS를 붓고 같이 볶아주면 끝!', '5. 짭조름한 깻잎나물을 먹는거 같기도하고 액젓이 이렇게 맛있는건지  이제야 깨달았습니다. 공심채랑 어울리니 감칠맛이 굿굿ㅎㅎ 새로운 맛을 경험해봤습니다. 볶음밥에 얹어먹으면 더 맛있을 듯 합니다.']
3. 녹두죽 - 재료: 녹두 1.5비율, 찹쌀 1비율, 들기름 약간, 소금 약간 - 레시피: ['1. 찹쌀을 깨끗이 씻어 불려줍니다.', '2. 찹쌀 보다 조금 더 많이 녹두를 준비해요. 찹쌀: 녹두 = 1: 1.5', '3. 녹두를 잘 씻어줍니다.', '4. 

In [5]:
print(f"총 저장된 FAISS 벡터 개수: {index.ntotal}")

# 첫 번째 벡터 출력
first_vector = index.reconstruct(0)
print("첫 번째 벡터:", first_vector[:50])  # 10개만 출력

총 저장된 FAISS 벡터 개수: 2071
첫 번째 벡터: [-0.00056066  0.00958696  0.01090676 -0.02947774 -0.00564767  0.02827183
 -0.02963853 -0.00037392 -0.03497132 -0.02007166  0.0015233   0.01633335
 -0.01782063 -0.00088852  0.00153837  0.01055839  0.02228249  0.02225569
  0.01238065 -0.02267106  0.00818677 -0.00416038 -0.01631995 -0.01556961
  0.00379526  0.00822027  0.00879643 -0.00931899  0.00384216  0.01051819
 -0.00151157 -0.0130171  -0.01708369 -0.0019864  -0.0104847   0.00324758
  0.01424311  0.00579171  0.00430107 -0.01341237 -0.00817337 -0.00712825
  0.01321139  0.0078853   0.00383546  0.01914713 -0.01266203 -0.03406019
 -0.01376075  0.03036207]
